# 01 · GSE65391 · RNA_array · metadata

Reads the sample characteristics from the series matrix. Writes `data/run_artifacts/GSE65391/metadata.rds`.

GEO stores each characteristic as `key: value`. `Not Applicable`, `Data Not Available`, `N/A` and `Unk.`
become missing values.

Derived fields:
- **stage**: SLEDAI category per visit: none 0, mild 1 to 5, moderate 6 to 10, high 11 to 19, very high 20
  or more.
- **nephritis_class**: the field arrives as two values joined by `;` (for example
  `Data Not Available;Prolif`); the class is the second value.
- **first_visit**: TRUE for the lowest visit number of each child.

## Read the sample characteristics

In [ ]:
source("/home/jovyan/session_data/Attempt0/src/paths.R")
suppressMessages({library(GEOquery); library(Biobase)})
pd <- pData(suppressMessages(getGEO(filename = raw("GSE65391", "GSE65391_series_matrix.txt.gz"), getGPL = FALSE)))
c(samples = nrow(pd), characteristic_fields = length(grep(":ch1$", names(pd))))

**Explanation**
- `source("../src/paths.R")` loads the locations and helpers (see notebook 00).
- `library(Biobase)` loads the Bioconductor package that defines the ExpressionSet, the object GEOquery
  returns (Huber W et al. Orchestrating high-throughput genomic analysis with Bioconductor. *Nat Methods*
  2015;12:115-121).
- `getGEO(filename = ..., getGPL = FALSE)` reads the series matrix downloaded in notebook 00. `getGPL = FALSE`
  skips the platform annotation, which is not needed here.
- `pData()` returns the sample table of the ExpressionSet: one row per sample, one column per field.
- `pd` holds that table.
- `grep(":ch1$", names(pd))` finds the positions of the columns whose names end in `:ch1`. GEOquery gives each
  characteristic such a column, for example `sledai:ch1`. `$` means "at the end".
- `c(samples = ..., characteristic_fields = ...)` prints the two counts with names.

## Two helper functions

In [ ]:
ch <- function(key) {
  v <- as.character(pd[[paste0(key, ":ch1")]])
  v[v %in% c("Not Applicable", "Data Not Available", "NA", "N/A", "Unk.", "")] <- NA
  v
}
num <- function(key) suppressWarnings(as.numeric(ch(key)))

**Explanation**
- `ch(key)` returns one characteristic as text, for example `ch("subject")`.
  - `paste0(key, ":ch1")` builds the column name, for example `subject:ch1`.
  - `pd[[...]]` takes that column; `as.character()` makes it text.
  - `v %in% c(...)` is TRUE for the values that mean "no data"; `<- NA` replaces them with a missing value.
  - The last line, `v`, is what the function returns.
- `num(key)` returns one characteristic as numbers, for example `num("sledai")`.
  - `as.numeric()` converts text to numbers.
  - `suppressWarnings()` hides the warning R gives when a value cannot be converted; such values become NA.

## Build the sample table

In [ ]:
meta <- data.frame(
  sample = rownames(pd), subject = ch("subject"),
  disease = factor(ch("disease state"), levels = c("Healthy", "SLE")),
  visit = as.integer(ch("visit")), batch = ch("batch"), set = ch("set"),
  cumulative_time = num("cumulative_time"),          # days since the child's first study visit
  age = num("age"), sex = ch("gender"), race = ch("race"),
  sledai = num("sledai"),
  nephritis_class = sub(".*;", "", ch("nephritis_class")),
  mdg = ch("mdg"),
  c3 = num("c3"), c4 = num("c4"), ds_dna = num("ds_dna"),
  wbc = num("wbc"), neutrophil_count = num("neutrophil_count"),
  lymphocyte_count = num("lymphocyte_count"), platelet_count = num("platelet_count"),
  oral_steroids = num("oral_steroids_category"), iv_steroids = num("steroid_iv_category"),
  mycophenolate = num("mycophenolate_category"), hydroxychloroquine = num("hydroxychloroquine_category"),
  cyclophosphamide = num("cyclophosphamide_category"),
  stringsAsFactors = FALSE, row.names = rownames(pd))

**Explanation**
- `data.frame(name = values, ...)` builds a table. Each `name = values` pair makes one column: the name on
  the left of `=` is the column name we choose, and the right side gives its values. For example,
  `sledai = num("sledai")` makes a column named `sledai` that holds the numbers read from the GEO field
  `sledai:ch1`, and `sex = ch("gender")` makes a column named `sex` from the GEO field `gender:ch1`.
- `sample = rownames(pd)`: the GEO sample identifier, for example `GSM1594271`.
- `factor(..., levels = c("Healthy", "SLE"))` stores disease as a category with Healthy first, so later
  comparisons read SLE against Healthy. For example, a model coefficient for disease is then the SLE value
  minus the Healthy value.
- `as.integer()` stores the visit number as a whole number. For example, the text `"3"` becomes the number 3.
- `sub(".*;", "", ...)` removes everything up to and including the last `;`, keeping the nephritis class.
  For example, `Data Not Available;Prolif` becomes `Prolif`.
- `stringsAsFactors = FALSE` keeps text columns as text.
- `row.names = rownames(pd)` names each row by its sample identifier, so rows can be found by sample. For
  example, `meta["GSM1594705", ]` returns the row for that sample.

## Add the derived fields

In [ ]:
meta$nephritis_class[meta$nephritis_class %in% c("Not Applicable", "Data Not Available")] <- NA
meta$stage <- cut(meta$sledai, c(-Inf, 0, 5, 10, 19, Inf),
                  labels = c("none", "mild", "moderate", "high", "very high"), ordered_result = TRUE)
# first visit: the lowest recorded visit number of each child; a sample with no visit number is not a first visit
meta$first_visit <- !is.na(meta$visit) &
  meta$visit == ave(meta$visit, meta$subject, FUN = function(v) if (all(is.na(v))) NA else min(v, na.rm = TRUE))
addmargins(table(disease = meta$disease, batch = meta$batch))

**Explanation**
- The first line sets the "no data" values left after `sub()` to missing.
- `cut(x, breaks, labels)` assigns each SLEDAI to an interval: 0; 1 to 5; 6 to 10; 11 to 19; 20 or more.
  Each interval includes its upper limit. For example, a SLEDAI of 5 is mild and 6 is moderate.
  `ordered_result = TRUE` keeps the order none, mild, moderate, high, very high.
- `ave(meta$visit, meta$subject, FUN = ...)` applies a function to each child's visits and returns one value
  per sample. The function returns the child's lowest visit number, or NA if the child has none. For
  example, a child with visits 1, 2 and 3 gets 1 on all three samples.
- `first_visit` is TRUE where the sample's visit number equals the child's lowest one. In the example, the
  sample with visit 1 is TRUE and the samples with visits 2 and 3 are FALSE. `!is.na(meta$visit) &`
  makes it FALSE for a sample with no visit number.
- `table(disease = ..., batch = ...)` counts samples by disease and array batch; `addmargins()` adds the
  totals.

## Count children and samples

In [ ]:
c(SLE_children = length(unique(meta$subject[meta$disease == "SLE"])),
  healthy_children = length(unique(meta$subject[meta$disease == "Healthy"])),
  SLE_samples = sum(meta$disease == "SLE"))
table(stage = meta$stage, useNA = "ifany")
table(nephritis_class = meta$nephritis_class, useNA = "ifany")
table(sex = meta$sex, disease = meta$disease)

**Explanation**
- `meta$subject[meta$disease == "SLE"]` takes the child identifier of every SLE sample; `unique()` keeps
  each child once; `length()` counts them.
- `sum(meta$disease == "SLE")` counts SLE samples: TRUE counts as 1 and FALSE as 0. For example,
  `sum(c(TRUE, FALSE, TRUE))` is 2.
- `table(..., useNA = "ifany")` counts each category and shows missing values as a separate count.

**Result.** 158 children with SLE contribute 924 samples; 46 healthy children contribute 72, of
which 24 are technical replicates.

## Check: samples with no visit number

In [ ]:
meta[is.na(meta$visit), c("subject", "disease", "sledai")]
table(visits_of_that_child = meta$subject %in% meta$subject[is.na(meta$visit)] & !is.na(meta$visit))

**Explanation**
- `meta[rows, columns]` selects part of the table. `is.na(meta$visit)` selects the samples with no visit
  number.
- The second line counts the other samples of those same children: TRUE would mean the child has other
  visits.

**Result.** One sample, GSM1594705 (child SLE-222), has no visit number, and it is that child's only
sample. It is not counted as a first visit, so first visits number 157, not 158.

## First visits of children with SLE

In [ ]:
fv <- meta[meta$disease == "SLE" & meta$first_visit & meta$set != "Technical_Replicate", ]
c(children = nrow(fv), unique_subjects = length(unique(fv$subject)))
summary(fv[, c("age", "sledai", "c3", "c4", "ds_dna", "neutrophil_count", "lymphocyte_count")])

**Explanation**
- These are the samples WGCNA uses: one sample per child with SLE.
- `fv` keeps the rows that are SLE, first visit, and not a technical replicate. `&` requires all three.
- `nrow(fv)` counts the samples; `length(unique(fv$subject))` counts the children. The two must be equal.
- `summary()` gives the minimum, quartiles, mean, maximum and number missing for each column.

## Save

In [ ]:
saveRDS(meta, art("GSE65391", "metadata.rds"))

**Explanation**
- `saveRDS(object, path)` saves one R object to a file. Later notebooks read it with `readRDS()`.
- `art("GSE65391", "metadata.rds")` gives the path `data/run_artifacts/GSE65391/metadata.rds` (see notebook 00).